# Play with pptx and "Config"-slides
in order to run the following code you should:
- have package python-pptx installed

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt, Cm
import re, os, sys
from pathlib import Path

REPORT_DIR = Path(os.getcwd())
#print(os.getcwd(),REPORT_DIR)

TEST_ROOT = REPORT_DIR.parent.parent.parent
if str(TEST_ROOT) not in sys.path:
    sys.path.append(str(TEST_ROOT))

from baseTestRoot import *

import Scriptum # type: ignore
Scriptum.__all__


In [ ]:
t = Scriptum.tag.createTag('color name=blubb uuu')
t.puretag, t.args

In [ ]:
template = Scriptum.ManagedPptx("template.pptx")
prs = template.document

In [ ]:
template.config

In [ ]:
template.config_layouts

In [ ]:
l=template.config_layouts['colors']
s=l.shapes[1]

In [ ]:
s.text_frame.paragraphs[0].runs[0].font.color.theme_color


In [ ]:
reportPptx.setFontColor(s.text_frame.paragraphs[0].runs[0],colorname='darkblue')

In [ ]:
from pptx.dml.color import RGBColor
type(RGBColor.from_string('112233'))

In [ ]:
import pptx
type(color.rgb) == pptx.dml.color.RGBColor
type(s.text_frame.paragraphs[0].runs[0].font)

In [ ]:
l=template.config_layouts['Tables']

In [ ]:
shape=l.shapes[3]
shape.has_table

In [ ]:
tables = {}
# try to catch tables
for shape in l.shapes:
    if not shape.has_table:
        continue
    table = shape.table
    # first cell contains paragraph with table name
    found = reportPptx.getSimpleTag(table.cell(0,0).text.lower(),'\<table name=[a-z0-9]+\>')
    
    if not found:
        continue
    name = found[1]['name']
    pars = {}
    # get font from cell(0,0)
    c = table.cell(0,0)
    tf = c.text_frame
    pars['default'] = reportPptx._extractFontAndDecorators(tf.paragraphs[0])
    pars['default']['alignment'] = tf.paragraphs[0].alignment
    pars['default']['verical_anchor'] = c.vertical_anchor
    
    for i,cell in enumerate(table.iter_cells()):
        if i == 0:
            continue # was already cell(0,0)
        found = reportPptx.getSimpleTag(cell.text.lower(),'\<[a-z0-9=_\s]+\>')
        if not found:
            continue
        tf = cell.text_frame
        pars[found[0]] = reportPptx._extractFontAndDecorators(tf.paragraphs[0])
        pars[found[0]]['alignment'] = tf.paragraphs[0].alignment
        pars[found[0]]['verical_anchor'] = cell.vertical_anchor
        
        tcPr = cell._tc.get_or_add_tcPr()
        styles = {}
        for c in tcPr.getchildren():
            if 'solidFill' in c.tag:
                color = reportPptx._getColorFromSolidFill(c)
                styles['solidFill'] = {'color': color}
            else:
                for tag in [ 'lnL', 'lnT', 'lnB', 'lnR' ]:
                    if tag in c.tag:
                        color = [ reportPptx._getColorFromSolidFill(subc) for subc in c.getchildren() 
                                 if 'solidFill' in subc.tag ]
                        if color and color[0][0] in ('theme','rgb'):
                            color = color[0]
                            width = dict(c.items()).get('w',0)
                            styles[tag] = {'color': color, 'width': width}
        pars[found[0]]['style'] = styles
    
    tables[name] = pars

print(tables)


In [ ]:
prs.save('table.pptx')

In [ ]:
cell = l.shapes[1].table.cell(2,0)

In [ ]:
from pptx.enum.dml import MSO_THEME_COLOR, MSO_FILL
for cell in l.shapes[1].table.iter_cells():
    
    #cell.fill.solid()
    #cell.fill.patterned()
    #cell.fill.back_color.rgb
    if cell.fill.type != MSO_FILL.BACKGROUND:
        cell.fill.solid()
        print(cell.text,cell.fill.fore_color.rgb)
        print(dir(cell.fill.fore_color))
        cell.fill.fore_color.rgb = RGBColor.from_string('112233')
        

In [ ]:
txt=cell._tc.get_or_add_txBody()

In [ ]:
firstpar=cell.text_frame.paragraphs[0]._p.r_lst[0].get_or_add_rPr()

In [ ]:
firstpar=p.get_or_add_rPr()

In [ ]:
firstpar.sz/100

In [ ]:
firstpar.latin.typeface

In [ ]:
pic._element._nvXxPr.cNvPr.attrib['title']
pic._element.spPr


In [ ]:
pfile= Presentation("Python_Jump-In_2019-JUL_vx.pptx")

In [ ]:
slide=pfile.slides[0]

In [ ]:
for shape in slide.shapes:
    print(type(shape))
    try:
        print(shape._element.xml)
    except:
        pass

In [ ]:
print(shape.element.xml)

In [ ]:
sub1=shape.shapes[1]

In [ ]:
print(sub1.element.xml)

In [ ]:

sub1.click_action.hyperlink.address ='file:///c:\\foo\\bar.mp4'

In [ ]:
pfile.save('new.pptx')